<a href="https://colab.research.google.com/github/EAduAgyei/EAduAgyei/blob/main/MyProject2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Please create a character-level language model on the Shakespeare dataset that extends our work in class:

1. Scale the model up.
2. Turn on the dropout mechanism.
3. Train the model on GPU.

-"Introduce more Multi-Head attention layers or double it"
- You are expected to create a new Colab notebook for the final project. --- Check!
- Do not include unnecessary code from this notebook. Copying everything into your work will hurt your scores.
- Record the training time (start, finish, and time elapsed).
- Generate 5000 characters from the model and analyze its performance.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

1. Scale the model up.

In [ ]:
batch_size = 72
max_iters = 5000
learning_rate = 5e-4
block_size = 32
eval_interval = 100
eval_iters = 200
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
n_embd = 768
n_head = 12
#head_size = 64
n_layer = 12

In [ ]:
t = torch.tensor(range(768))
print(t)
print(t.shape)

t2 = t.reshape(64, 12)
print(t2)
print(t2.shape)

tensor([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
         14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
         28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
         42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
         56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
         70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
         84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
         98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
        112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
        126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
        140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
        154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167,
        168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 1

In [ ]:
torch.manual_seed(1337)

In [ ]:
! wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

--2025-08-10 23:23:25--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2025-08-10 23:23:25 (23.3 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print("Vocabulary Size:", vocab_size)
print("Mapping:")
print(stoi)

Vocabulary Size: 65
Mapping:
{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [ ]:

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

1003854 111540


In [ ]:
def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]).to(device)
    return x, y

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

2. Turn on the dropout mechanism.


In [ ]:
dropout = 0.3

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.proj2 = nn.Linear(n_embd, n_embd)
        self.proj3 = nn.Linear(n_embd, n_embd)
        self.proj4 = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.proj2(out)
        out = self.proj3(out)
        out = self.proj4(out)
        out = self.dropout(out)
        return out

In [ ]:
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):

        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class AbyssModel(nn.Module):

  def __init__(self, vocab_size, batch_size, n_embd, n_head, n_layer):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)


  def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

  def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
          idx_cond = idx[:, -block_size:]
          logits, loss = self(idx_cond)
          logits = logits[:, -1, :]
          probs = F.softmax(logits, dim=-1)
          idx_next = torch.multinomial(probs, num_samples=1)
          idx = torch.cat((idx, idx_next), dim=1)
        return idx



3. Train the model on GPU. (Record the training time (start, finish, and time elapsed).
Generate 5000 characters from the model and analyze its performance.)

In [ ]:
model = AbyssModel(vocab_size, batch_size, n_embd, n_head, n_layer).to(device)

def train_model(model):

  optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

  for iter in range(max_iters):

        xb, yb = get_batch('train')
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if iter % eval_interval == 0 or iter == max_iters - 1:
         losses = estimate_loss()
         print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")


In [ ]:
train_model(model)

step 0: train loss 6.3816, val loss 6.4777
step 100: train loss 2.3470, val loss 2.3452
step 200: train loss 2.1378, val loss 2.1819
step 300: train loss 2.0067, val loss 2.0858
step 400: train loss 1.9163, val loss 2.0283
step 500: train loss 1.8444, val loss 1.9739
step 600: train loss 1.7844, val loss 1.9286
step 700: train loss 1.7478, val loss 1.9076
step 800: train loss 1.7126, val loss 1.8778
step 900: train loss 1.6861, val loss 1.8654
step 1000: train loss 1.6677, val loss 1.8370
step 1100: train loss 1.6363, val loss 1.8148
step 1200: train loss 1.6230, val loss 1.7978
step 1300: train loss 1.5974, val loss 1.7853
step 1400: train loss 1.5887, val loss 1.7789
step 1500: train loss 1.5699, val loss 1.7536
step 1600: train loss 1.5627, val loss 1.7553
step 1700: train loss 1.5481, val loss 1.7396
step 1800: train loss 1.5328, val loss 1.7342
step 1900: train loss 1.5259, val loss 1.7142
step 2000: train loss 1.5177, val loss 1.7080
step 2100: train loss 1.5080, val loss 1.7108


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=5000)[0].tolist()))


I will have our Citizenes: three poor spates compeace and part? let's have
better too cruel, and that of hopein to bad her
Of Herefort, vallet bad woman be
O'er I degree, and his fac crease:
For I drown 'embray you and lost that I, who
Made all no people she wail I roes to all like your naddifice,
Believe abhorm'd his own honour,
Be good son here as the favour go
Dozens mocented two fray-maid or England: to he he will sworn this condite, with I,
How for the earth of state too deadlen bubt,
And 
rules tell thee againtly squears?

Alluce:

Nurse:
When to brait if I to claim the man;
For obey-slaily thereby, Warwick nobles, Ambious master.

BUCKINGHAM:
one noble less; I pray your promise,
When she have bequeen fray some fifters tempresed:
If debeles on thy hazined love: I'll be home you
Why subssion jewels are a justice.

GLOUCESTER:
How ply I fair.

CORIOLAUGHBY:

Doom the night of high weeps; and we shall make me for time?

QUEEN ELIZABETH:
An afore good brother drebled!

EARL:
ANGELOU

8/9/25

1st time lapse training m (through while model build): 34m22s
- Time start: 5:47pm
- Time completed: 6:22pm

2nd time lapse training m: 18m26s
- Time start: 10:50pm
- Time completed: 11:09pm

8/10/25

3rd time lapse training m: 1hr22m
- Time start: 12:14pm
- Time completed:1:52pm

4th time lapse training m: 1hr37m
- Time start: 2:03pm
- Time completed: 3:40pm

5th time lapse training m: 1hr12m
- Time start: 7:23pm
- Time completed: 9:11pm




The new model I created requires more GPU usage, which made hard for me run the codes. So I have get a subscription for Colab for an extension.

There's an free offer for students for a year.

As for its performance, it has managed to create a whole script despite the entire dialogue being complete gibberish at first. In a closer look, the capitial letters that were meant for the names in the script have lower case letters in them.

However with the following training retries (2nd and 3rd), the text became more gibberish, even with increasing the dropbox a little.

So I keep training the model to the point it does a proper job with the dialogues while adjusting the sizes, dropout, and learning rate.

Fourth try has a complete dialogue now, just a few names being uncapitalized and a few lines were missing.

Fifth try, I increased the learning rate  and batch size to make the training faster and smoother for the dialogue. In result, the model has finally generated full sentences.

So what I've learned through creating the model is the bigger the model scaling components, the quicker the training will be. That way the model can take its time gathering the entire language and script.